## Task 1 ##

Please press "q" key to exit video.

In [ ]:
import cv2
import numpy as np
from scipy.spatial import distance

# Load video file
video_path = "Traffic_Laramie_1.mp4"
cap = cv2.VideoCapture(video_path)

# Initialize background subtractor
fgbg = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=50, detectShadows=True)

# Define kernel
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

# Read the first frame for frame differencing
ret, prev_frame = cap.read()
if not ret:
    print("Error: Unable to read video")
    cap.release()
    exit()

# Convert first frame to grayscale
prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

# Define the starting position of the detection area (bottom 55% of the screen)
roi_start = int(prev_gray.shape[0] * 0.45)

# Dictionary to track cars {ID: (x, y, w, h, stationary_frames)}
tracked_cars = {}
next_car_id = 0
max_stationary_frames = 180  # Assuming 30 FPS, 6 seconds means 180 frames

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Convert current frame to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Select Only the Bottom 55%
    roi = gray[roi_start:, :]
    roi_frame = frame[roi_start:, :]

    # Frame Differencing
    prev_roi = prev_gray[roi_start:, :]
    frame_diff = cv2.absdiff(prev_roi, roi)  # Compute difference between consecutive frames
    _, frame_diff = cv2.threshold(frame_diff, 25, 255, cv2.THRESH_BINARY)

    # Background Subtraction
    fgmask = fgbg.apply(roi)

    # Combine both techniques
    combined_mask = cv2.bitwise_or(fgmask, frame_diff)

    # Remove noise
    combined_mask = cv2.morphologyEx(combined_mask, cv2.MORPH_OPEN, kernel, iterations=2)
    combined_mask = cv2.morphologyEx(combined_mask, cv2.MORPH_DILATE, kernel, iterations=2)

    # Find contours
    contours, _ = cv2.findContours(combined_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    detected_cars = []

    # Process detected objects
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > 1200:  # Ignore small rectangles (anything below 1200 pixels)
            x, y, w, h = cv2.boundingRect(cnt)
            y_full = y + roi_start  # Convert y position to full-frame coordinates

            # Exclude People
            aspect_ratio = w / float(h)
            if 1.5 < aspect_ratio < 4.5:  # Typical aspect ratio for cars
                detected_cars.append((x, y_full, w, h))

    # Match Detected Cars to Existing Tracked Cars
    updated_tracked_cars = {}
    used_ids = set()

    for x, y, w, h in detected_cars:
        matched = False
        for car_id, (tx, ty, tw, th, stationary_frames) in tracked_cars.items():
            dist = distance.euclidean((x, y), (tx, ty))
            if dist < 50 and car_id not in used_ids:  # If detected car is close to a previously tracked car
                if (x, y) == (tx, ty):  # If the car has stopped
                    stationary_frames += 1
                else:
                    stationary_frames = 0

                updated_tracked_cars[car_id] = (x, y, w, h, stationary_frames)
                used_ids.add(car_id)
                matched = True
                break
        
        if not matched:
            updated_tracked_cars[next_car_id] = (x, y, w, h, 0)
            used_ids.add(next_car_id)
            next_car_id += 1

    # Remove cars that left the 55% region
    tracked_cars = {
        car_id: (x, y, w, h, stationary_frames)
        for car_id, (x, y, w, h, stationary_frames) in updated_tracked_cars.items()
        if y + h > roi_start  # Remove cars that left the detection area
    }

    # Draw 55% Detection Boundary Line
    for i in range(0, frame.shape[1], 20):  # Draw a dashed red line
        cv2.line(frame, (i, roi_start), (i + 10, roi_start), (0, 0, 255), 2)

    # Add "Main Street" label above the line
    cv2.putText(frame, "Main Street", (10, roi_start - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    # Draw Bounding Boxes for Tracked Cars
    for car_id, (x, y, w, h, stationary_frames) in tracked_cars.items():
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)  # Keep the box fixed on detected cars

    # Display the processed frame
    cv2.imshow("Car Detection", frame)

    # Update previous frame for next iteration
    prev_gray = gray.copy()

    # Break loop on 'q' key press
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()